## Reading Bronze.customers Delta Table

In [0]:
customers_bronze_path = "s3://travel-analytics-bronze/delta/bronze/customers/"
customers_bronze_df = spark.read.format("delta").load(customers_bronze_path)

## Silver Transformations

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, trim, upper, to_date, to_timestamp, coalesce, lit, initcap, when
)

# =============================================================
# STEP 0: CONFIGURATION & SETUP
# =============================================================
table_name = "customers"
customers_silver_path = f"s3://travel-analytics-bronze/delta/silver/{table_name}/"

# =============================================================
# STEP 1: DATA TYPE CASTING & PARSING
# =============================================================
print("\nSTEP 1: Casting Data Types (Customers)...")

customers_step_1_df = (
    customers_bronze_df

    # Numeric / ID columns
    .withColumn("id", col("id").cast("int"))

    # String columns: trim and standardize
    .withColumn("email", trim(col("email")))
    .withColumn("gender", upper(trim(col("gender"))))
    .withColumn("country", initcap(trim(col("country"))))
    .withColumn("first_name", initcap(trim(col("first_name"))))
    .withColumn("family_name", initcap(trim(col("family_name"))))
    .withColumn("phone_number", trim(col("phone_number")))

    # Birth date parsing
    .withColumn("birth_date", to_date(col("birth_date.member0")))

    # CDC updated timestamp
    .withColumn("updated_at", to_timestamp(col("_ab_cdc_updated_at")))
)

# =============================================================
# STEP 2: CLEANING & BUSINESS LOGIC
# =============================================================
print("\nSTEP 2: Cleaning & Standardization (Customers)...")

customers_step_2_df = (
    customers_step_1_df

    # Fill missing string columns with UNKNOWN
    .withColumn("email", coalesce(col("email"), lit("UNKNOWN")))
    .withColumn("gender", coalesce(col("gender"), lit("UNKNOWN")))
    .withColumn("country", coalesce(col("country"), lit("UNKNOWN")))
    .withColumn("first_name", coalesce(col("first_name"), lit("UNKNOWN")))
    .withColumn("family_name", coalesce(col("family_name"), lit("UNKNOWN")))
    .withColumn("phone_number", coalesce(col("phone_number"), lit("UNKNOWN")))

    # Replace null birth_date with default date 1999-01-01
    .withColumn("birth_date", coalesce(col("birth_date"), lit("1999-01-01").cast("date")))
)

# =============================================================
# STEP 3: DEDUPLICATION & DROP AIRBYTE METADATA
# =============================================================
print("\nSTEP 3: Deduplication & Dropping Airbyte Columns (Customers)...")

airbyte_columns_to_drop = [
    "_airbyte_ab_id",
    "_airbyte_emitted_at",
    "_airbyte_additional_properties",
    "_ab_cdc_lsn",
    "_ab_cdc_deleted_at",
    "_ab_cdc_updated_at"
]

customers_step_3_df = (
    customers_step_2_df

    # Deduplicate based on id
    .dropDuplicates(["id"])

    # Drop Airbyte metadata
    .drop(*airbyte_columns_to_drop)
)

# =============================================================
# STEP 4: BUSINESS-FRIENDLY COLUMN NAMES
# =============================================================
print("\nSTEP 4: Renaming Columns (Customers)...")

rename_map = {
    "id": "Customer_Id",
    "email": "Email",
    "gender": "Gender",
    "country": "Country",
    "birth_date": "Birth_Date",
    "first_name": "First_Name",
    "family_name": "Family_Name",
    "phone_number": "Phone_Number",
    "age": "Age",
    "updated_at": "Updated_At"
}

customers_silver_df = customers_step_3_df.select(
    [col(c).alias(rename_map.get(c, c)) for c in customers_step_3_df.columns]
)



STEP 1: Casting Data Types (Customers)...

STEP 2: Cleaning & Standardization (Customers)...

STEP 3: Deduplication & Dropping Airbyte Columns (Customers)...

STEP 4: Renaming Columns (Customers)...


In [0]:
customers_silver_df.display()

Customer_Id,Email,Gender,Country,Birth_Date,First_Name,Family_Name,Phone_Number,Updated_At
12801,marieannick.viemont.12801@company.com,F,Germany,1989-07-15,Marie Annick,Viemont,492041525792,2025-12-12T01:04:52.921Z
16419,christiane.bellard.16419@company.com,M,Austria,1979-05-17,Christiane,Bellard,437393149798,2025-12-12T01:04:52.921Z
39098,bernardette.piou.39098@company.com,F,Colombia,1977-05-22,Bernardette,Piou,5754979937,2025-12-12T01:04:52.921Z
40241,bachir.audigane@example.com,M,Japan,1986-01-04,Bachir,Audigane,818018615452,2025-12-12T01:04:52.921Z
44330,mariehelene.ripoche.44330@company.com,F,United States Of America,1988-10-13,Marie Helene,Ripoche,18280926116,2025-12-12T01:04:52.921Z
59700,leon.pithon.59700@company.com,M,Iran,1988-01-17,Leon,Pithon,980377976117,2025-12-12T01:04:52.921Z
73552,vinciane.arnou.73552@company.com,F,Austria,1981-03-16,Vinciane,Arnou,439066955202,2025-12-12T01:04:52.921Z
86899,charline.laurendeau@example.com,F,Japan,1974-06-26,Charline,Laurendeau,814979490203,2025-12-12T01:04:52.921Z
92645,denise.cognee.92645@company.com,F,Austria,1989-02-20,Denise,Cognee,433624022947,2025-12-12T01:04:52.921Z
111049,samir.mechineau@example.com,M,India,1981-10-12,Samir,Mechineau,914442544587,2025-12-12T01:04:52.921Z


## Writing Silver customer to Delta Lake 

In [0]:
# =============================================================
# STEP 5: Persist Silver Table
# =============================================================
print("\nSTEP 5: Persist Customers Silver Table...")
table_name = "customer"
customers_silver_path = f"s3://travel-analytics-bronze/delta/silver/{table_name}/"

customers_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(customers_silver_path)

print(f" Customers table saved to Silver layer at {customers_silver_path}")


STEP 5: Persist Customers Silver Table...
 Customers table saved to Silver layer at s3://travel-analytics-bronze/delta/silver/customer/
